## Model Selection

Before we move on we would like to choose the best possible model for each of the two cases: daily forecast and hourly forecast. It seems that for the daily forecast in the long term you choose linear_order2 in the short to medium term potentially hybrid_order2. For the hourly forecast it seems that just using XGBoost is the best possible model.

The main question is that in the daily forecast if we were to optimise the hyperparameters of the XGBoost model within the hybrid model would it make hybrid_order2 better than linear_order2 in both the short and long term. It would also be nice to do hyperparamter optimisation for the hourly forecast as well just to see whether we can improve the predictions or not. 

Finally it would be good to look at SHAP values to see if we can drop any of the features, particualarly some of the lags as they can make the models computationally expensive.

For our own use (maybe delete later): http://kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

https://hyperopt.github.io/hyperopt/?source=post_page

https://github.com/hyperopt/hyperopt/wiki/FMin

In [ ]:
from hyperopt import hp, fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import xgboost as xgb
import pickle
import pandas as pd
from jfk_taxis import load_design, load_models, load_lags, run_forecasts, preprocess, forecast, create_val_data, wrapped_objective
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import time
import numpy as np
import cupy as cp

In [2]:
# First reload the significant lags
daily_lags = load_lags("daily", "eda")

hourly_lags = load_lags("hourly", "eda")

In [3]:
# Get both the full daily and hourly time series
dir_path = "../data/processed/"
df_daily = pd.read_csv(f"{dir_path}ts_daily2011-2025.csv")
df_hourly = pd.read_csv(f"{dir_path}ts_hour2011-2025.csv")

# Convert dates to datetime objects
df_daily["pickup_date"] = pd.to_datetime(df_daily["pickup_date"])
df_hourly["dt"] = pd.to_datetime(df_hourly["dt"])


In [4]:
# To pass the time series through our helper functions they need to be a pandas series indexed by a datetime object:
ts_hourly = df_hourly["trips"]
ts_hourly.index = df_hourly["dt"]

ts_daily = df_daily["trips"]
ts_daily.index = df_daily["pickup_date"]

In [5]:
# We now need to split into test and train data, we will train on the pre 2024 data and test on 2024 onwards, approx a 90:10 split
ts_daily_train = ts_daily[:"2023-12-31"]
ts_daily_test = ts_daily["2024-01-01":]

ts_hourly_train = ts_hourly[:"2023-12-31"]
ts_hourly_test = ts_hourly["2024-01-01":]

In [ ]:
# Define search space
space = {
    # We rely on early stopping when fitting so this isn't an optimised value
    # number of trees
    "n_estimators": 500,

    # Learning rate
    # step size shrinkage, smaller = slower but more precise learning
    "learning_rate": 0.05,

    # Depth/complexity
    # Max depth of tree, larger more complex trees but can cause overfitting
    "max_depth": scope.int(hp.quniform("max_depth", 3, 6, 1)), # scope.int ensures we take ints only
    # minimum "weight" needed in child node. Higher values more conservative, fewer splits helps prevent overfitting
    "min_child_weight": hp.loguniform("min_child_weight", -2.3, 2.3), # approx [0.1, 10]

    # Randomisation/feature subsampling
    # fraction of rows used per tree, lower adds randomness reduces overfitting
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    # fraction of features used per tree
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),

    # Regularisation
    # L2 penalty, good range is [0.1, 10] we use loguniform because this means that every order of magnitude has equal probability, the def of log uniform in hyperopt is that it returns a value exp(U(low,high)) where U is uniform dist.  
    "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-2), np.log(100)), # [0.01, 100]
    # L1 penalty
    "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-3), np.log(10)), # [0.001, 10]

    # Split pnealty (gamma) 
    # minimum loss reduction required to split a node, higher values = more conservative
    "gamma": hp.loguniform("gamma", -7.0, 2.3), # approx [0.0009, 10]

    "random_state": 37,
    #"early_stopping_rounds": 100,
    "eval_metric": "mae", 
    "tree_method": "hist",
    "device": "cuda" # Use GPU if available
}
    
    

In [14]:
# Set the parameters for creat_val_data, daily ts
n_splits = 5
test_size = 30
lags = daily_lags
constant = False
order = 0
fourier_features = ["YE", "W"]
time_step = "D"
ts = ts_daily_train

# Set the parameters for objective
steps = 30

In [15]:
# Set the parameters for creat_val_data, hourly ts
n_splits = 5
test_size = 168
lags = hourly_lags[:168]
constant = False
order = 0
fourier_features = ["D", "W"]
time_step = "h"
ts = ts_hourly_train

# Set the parameters for objective
steps = 168

In [16]:
# Create the folds
fold_dict = create_val_data(n_splits, test_size, lags, constant, order, fourier_features, time_step, ts)

Fold 0
[   0    1    2 ... 4595 4596 4597]
Fold 1
[   0    1    2 ... 4625 4626 4627]
Fold 2
[   0    1    2 ... 4655 4656 4657]
Fold 3
[   0    1    2 ... 4685 4686 4687]
Fold 4
[   0    1    2 ... 4715 4716 4717]


In [ ]:
# Set attributes of wrapped_objective
wrapped_objective.fold_dict = fold_dict
wrapped_objective.lags = lags
wrapped_objective.steps = steps


In [ ]:
# Optimisation algorithm
trials = Trials()

best_hyperparams = fmin(fn = wrapped_objective,
                        space = space,
                        algo = tpe.suggest,
                        max_evals = 100,
                        trials = trials)

(4227, 360)                                            
Fit time: 4.38 seconds                                 
Predict time: 0.4280 seconds                           
(4257, 360)                                            
Fit time: 4.24 seconds                                 
Predict time: 0.3566 seconds                           
(4287, 360)                                            
Fit time: 4.09 seconds                                 
Predict time: 0.3575 seconds                           
(4317, 360)                                            
Fit time: 4.13 seconds                                 
Predict time: 0.3533 seconds                           
(4347, 360)                                            
Fit time: 4.16 seconds                                 
Predict time: 0.3909 seconds                           
MAEs:                                                  
[432.060595703125, 1012.4962727864583, 354.4336263020833, 437.52159830729164, 674.14912109375]
Avg MAE: 

In [ ]:
print("The best hyperparamters are: ", "\n")
print(best_hyperparams)

In [ ]:
# It would now be interesting to use this hyperparams and use them on the forecasts from the previous notebook to see how they compare

# Load the previous non linear model
linear_models_loaded, non_linear_models_loaded = load_models("5_order_linear_daily")

# Load the previous non linear design matrix
linear_design_loaded, non_linear_design_loaded = load_design("5_order_linear_daily")

# Get design, target and dp
X = non_linear_design_loaded["base_non_linear"][0]
y = non_linear_design_loaded["base_non_linear"][1]
dp = non_linear_design_loaded["base_non_linear"][2]

# Now create our new non linear model trained on the full training data set
new_non_linear = xgb.XGBRegressor(
        n_estimators = best_hyperparams["n_estimators"],
        learning_rate = best_hyperparams["learning_rate"],
        max_depth = best_hyperparams["max_depth"],
        min_child_weight = best_hyperparams["min_child_weight"],
        subsample = best_hyperparams["subsample"],
        colsample_bytree = best_hyperparams["colsample_bytree"],
        gamma = best_hyperparams["gamma"],
        reg_alpha = best_hyperparams["reg_alpha"],
        reg_lambda = best_hyperparams["reg_lambda"],
        random_state = 37,
        eval_metric = "mae",
        tree_method = "hist",
        device = "cuda"
        )


new_non_linear.fit(X, y,
    verbose = False)

non_linear_models_loaded["new_non_linear"] = (new_non_linear, dp, None)

In [ ]:
# Steps for the forecast
steps = [1, 2, 3, 7, 14, 28, 30, 60, 180, 365, 500, 546]


In [ ]:
# Run forecasts
run_forecasts(steps, daily_lags, {}, non_linear_models_loaded, False, "D", ts_daily_train, ts_daily_test)